In [1]:
pip install lightkurve pandas matplotlib batman-package --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.1/261.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 49.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fssp

In [2]:
import pandas as pd #data analysis library loads ASTAP .CSV to strucutred table (DataFrame)
import lightkurve as lk # Time series astrophotometry
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
import batman

/usr/local/lib/python3.13/dist-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/ag25-lgtm/Astrophotometry-Differential-Light-Curve/refs/heads/main/HJD_bjd_tdb.csv") #Loads .csv file

time = df["BJD_TDB"] #extracts timestap column of Heliocentric julian data
v_mag = df["053454.0-040637"] #extracts magnitude measurments of target variable star
c_mag = df["053432.5-042915"] #extracts magnitude measurments of comparison star
k_mag = df["053459.0-042115"] #extracts magnitude measurments of check star

v_minus_c = v_mag - c_mag #subtracts comparision star mag from variable mag to find differential magnitude due to star variability
k_minus_c = k_mag - c_mag #subtracts comparision star from check star to ensure comparison star is non-variable (ideally constant)

#convert differential magnitude (log) into relative flux ratio (linear) by Pogson's Relation
rel_v_flux = 10 ** (-0.4 * v_minus_c) #calculate relative flux of variable star to comparision star
rel_k_flux = 10 ** (-0.4 * k_minus_c) #calculate relative flux of check star to comparision star

#Create LightKurve Objects
lc_var = lk.LightCurve(time=time, flux=rel_v_flux, label="Variable* (F_v / F_c)") #varaible star flux/time object
lc_var_clean = lc_var.remove_outliers(sigma=3) #removes outliers more than 3 sigma from the mean
lc_var_norm = lc_var_clean.normalize() #normalised
lc_var_binned = lc_var_norm.bin(time_bin_size=0.004).remove_nans() #averages cleaned variable star data into time chunks and averages

lc_check = lk.LightCurve(time=time, flux=rel_k_flux, label="Check* (F_k / F_c) - 0.13") #check star flux/time object
lc_check_clean = lc_check.remove_outliers(sigma=3)
lc_check_norm = lc_check_clean.normalize() #normalised
lc_check_shift = lc_check_norm - 0.13
lc_check_binned = lc_check_shift.bin(time_bin_size=0.004).remove_nans() #.remove_nans() purges invalid entries allowing the gap to be interpolated

#Plot Scatters on Single graph
var_graph = lc_var_norm.scatter(c='red', s=30) #create a scatter graph or the varaible flux/time object
lc_check_shift.scatter(ax=var_graph, c='blue', s=30) #scatter plots the check star flux/time on the same axes as variable

#Interpolating Gaps in data
def plot_gapped_line(var_graph, time, flux, color='black', label=None): #Defines a reusable function that can be used to plot any time and flux values
  dt = np.diff(time) #calculates the time difference btween sucessive data points
  gap_thresh = 2 * np.median(dt) #set a threshold to be 2 * median gap
  gap_indices = np.where(dt > gap_thresh)[0] #evaluates teh data indices for which the threshold is met

  start_idx = 0 #first index is 0
  for idx in gap_indices:
    var_graph.plot(time[start_idx:idx + 1], flux[start_idx:idx + 1], color='black', linewidth=1.5, label=label) #plots solid lines from the start upto the gap index
    var_graph.plot(time[idx:idx + 2], flux[idx:idx + 2], linestyle='--', color='black', linewidth=1.5, label=None) #plots dashed line from idx to idx +1
    start_idx = idx + 1 # repeats for the next point after the gap
  var_graph.plot(time[start_idx:], flux[start_idx:], color='black', linewidth=1.5)

#define variables
t_var = lc_var_binned.time.value #raw time values
t_check = lc_check_binned.time.value
f_var = lc_var_binned.flux.value #raw flux value
f_check = lc_check_binned.flux.value

#Call the defined function for the variable and check star
plot_gapped_line(var_graph, t_var, f_var, label="Binned 0.004")
plot_gapped_line(var_graph, t_check, f_check)

#Label Axes, gridlines etc
var_graph.set(xlabel="BJD_TDB", ylabel="Norm Rel Flux Ratio to Comp Star", title="V1045 Ori Differential Light Curve")
var_graph.set_ylim(0.815, 1.09)
plt.grid(True, which='major', linestyle="-", linewidth=0.75, alpha=1)
plt.grid(True, which='minor', linestyle="-", linewidth=0.75, alpha=0.4)

plt.legend(loc="upper right", fontsize=8)
plt.savefig("lightcurve.png", dpi=300)
files.download("lightcurve.png")
plt.show()